# Documentation QA with a transformer retriever: evaluation, ablations and demo UI

**Before running:** Runtime > Change runtime type > **T4 GPU**. Run the cells one by one (not "Run all") the first time, because the Drive popup needs a click.
This notebook is self-contained: it writes all project files to `MyDrive/doc-qa`, reuses the chunks and embeddings from an earlier run if they exist, and saves every result to `MyDrive/doc-qa/results`.

In [ ]:
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT = '/content/drive/MyDrive/doc-qa'
except Exception as e:
    print('Drive mount failed, using temporary storage instead:', e)
    PROJECT = '/content/doc-qa'
for d in ('src', 'data', 'results'):
    os.makedirs(f'{PROJECT}/{d}', exist_ok=True)
os.chdir(PROJECT)
print('Working in', os.getcwd())

In [ ]:
!pip install -q sentence-transformers rank_bm25 gradio

## 1. Project files

In [ ]:
%%writefile src/ingest.py
"""Split the FastAPI Markdown docs into heading-based chunks -> data/chunks.jsonl

Setup (run once from the repo root):
    git clone --depth 1 https://github.com/fastapi/fastapi data/fastapi
Then:
    python src/ingest.py
"""
import json
import re
from pathlib import Path

DOCS_DIR = Path("data/fastapi/docs/en/docs")
OUT = Path("data/chunks.jsonl")
MAX_CHARS = 1200   # split sections longer than this at paragraph boundaries
MIN_CHARS = 80     # drop tiny fragments

# Pages that are not documentation of the framework itself. release-notes.md alone is ~40% of all
# chunks and is full of keywords (404, response_model, ...) that pull it above the real docs.
EXCLUDE_FILES = {"release-notes.md", "fastapi-people.md", "external-links.md", "newsletter.md",
                 "management.md", "management-tasks.md", "contributing.md", "help-fastapi.md",
                 "translations.md", "translation-banner.md", "_llm-test.md", "agent-banner.md"}
EXCLUDE_DIRS = {"about", "resources", "css", "js", "img"}


def clean_heading(h):
    """'Query Parameters { #query-parameters }' -> 'Query Parameters'"""
    return re.sub(r"\s*\{\s*#[^}]*\}\s*$", "", h).strip()


def is_excluded(rel):
    return rel in EXCLUDE_FILES or rel.split("/")[0] in EXCLUDE_DIRS


def split_by_headings(text):
    """Return [(heading, body)] splitting on #, ## and ### (ignores headings inside code fences)."""
    sections, heading, buf, in_code = [], "", [], False
    for line in text.splitlines():
        if line.strip().startswith("```"):
            in_code = not in_code
        if not in_code and re.match(r"^#{1,3}\s", line):
            if buf:
                sections.append((heading, "\n".join(buf).strip()))
            heading, buf = clean_heading(line.lstrip("# ").strip()), []
        else:
            buf.append(line)
    if buf:
        sections.append((heading, "\n".join(buf).strip()))
    return sections


def split_long(text, max_chars):
    """Greedy paragraph packing so no chunk is (much) longer than max_chars."""
    chunks, cur = [], ""
    for para in text.split("\n\n"):
        if cur and len(cur) + len(para) > max_chars:
            chunks.append(cur.strip())
            cur = ""
        cur += para + "\n\n"
    if cur.strip():
        chunks.append(cur.strip())
    return chunks


def strip_front_matter(text):
    return re.sub(r"^---\n.*?\n---\n", "", text, count=1, flags=re.S)


def build_chunks(max_chars=MAX_CHARS, docs_dir=DOCS_DIR, exclude=True, with_title=True):
    """Return the list of chunk dicts.

    exclude:    drop non-documentation pages (see EXCLUDE_FILES / EXCLUDE_DIRS)
    with_title: store the page title (first H1) on every chunk so retrieval knows which page it is from
    """
    if not docs_dir.exists():
        raise SystemExit(f"{docs_dir} not found. Clone the repo first (see top of file).")
    chunks = []
    for path in sorted(docs_dir.rglob("*.md")):
        rel = path.relative_to(docs_dir).as_posix()
        if exclude and is_excluded(rel):
            continue
        text = strip_front_matter(path.read_text(encoding="utf-8"))
        m = re.search(r"^#\s+(.+)$", text, flags=re.M)
        title = clean_heading(m.group(1)) if m else ""
        for heading, body in split_by_headings(text):
            for piece in split_long(body, max_chars):
                if len(piece) < MIN_CHARS:
                    continue
                rec = {"id": len(chunks), "source": rel, "heading": heading, "text": piece}
                if with_title:
                    rec["title"] = title
                chunks.append(rec)
    return chunks


def main():
    chunks = build_chunks()
    OUT.parent.mkdir(parents=True, exist_ok=True)
    with OUT.open("w", encoding="utf-8") as f:
        for rec in chunks:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
    print(f"Wrote {len(chunks)} chunks to {OUT}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/retrieve.py
"""BM25 baseline and dense (transformer) retriever over data/chunks.jsonl

Quick test:
    python src/retrieve.py "How do I declare query parameters?"
"""
import hashlib
import json
import re
import sys
from pathlib import Path

import numpy as np

CHUNKS = Path("data/chunks.jsonl")
EMB_MODEL = "BAAI/bge-small-en-v1.5"
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "  # recommended for bge models


def load_chunks(path=CHUNKS):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f]


def _tok(text):
    return re.findall(r"\w+", text.lower())


def _doc_text(c):
    """Text that is indexed: 'Page title > Section heading' followed by the chunk text."""
    title, heading = c.get("title", ""), c.get("heading", "")
    head = f"{title} > {heading}" if title and heading and title != heading else (heading or title)
    return f"{head}\n{c['text']}"


class BM25Retriever:
    name = "bm25"

    def __init__(self, chunks):
        from rank_bm25 import BM25Okapi
        self.chunks = chunks
        self.bm25 = BM25Okapi([_tok(_doc_text(c)) for c in chunks])

    def search(self, query, k=5):
        scores = self.bm25.get_scores(_tok(query))
        top = np.argsort(scores)[::-1][:k]
        return [dict(self.chunks[i], score=float(scores[i])) for i in top]


class DenseRetriever:
    name = "dense"

    def __init__(self, chunks, model_name=EMB_MODEL):
        from sentence_transformers import SentenceTransformer
        self.chunks = chunks
        self.model = SentenceTransformer(model_name)
        texts = [_doc_text(c) for c in chunks]
        # The cache file name depends on the exact indexed text + model, so changing the chunks
        # (or the model) can never load stale embeddings.
        key = hashlib.md5((model_name + "\n".join(texts)).encode()).hexdigest()[:10]
        cache = Path(f"data/emb_{key}.npy")
        if cache.exists():
            self.emb = np.load(cache)
        else:
            self.emb = self.model.encode(texts, batch_size=32, normalize_embeddings=True,
                                         show_progress_bar=True)
            cache.parent.mkdir(exist_ok=True)
            np.save(cache, self.emb)

    def search(self, query, k=5):
        q = self.model.encode([QUERY_PREFIX + query], normalize_embeddings=True)[0]
        scores = self.emb @ q   # cosine similarity (vectors are normalized)
        top = np.argsort(scores)[::-1][:k]
        return [dict(self.chunks[i], score=float(scores[i])) for i in top]


class HybridRetriever:
    """Reciprocal Rank Fusion of BM25 and dense rankings (both must index the same chunks)."""
    name = "hybrid"

    def __init__(self, bm25, dense, rrf_k=60, pool=30):
        self.bm25, self.dense, self.rrf_k, self.pool = bm25, dense, rrf_k, pool

    def search(self, query, k=5):
        fused = {}
        for retriever in (self.bm25, self.dense):
            for rank, h in enumerate(retriever.search(query, k=self.pool), 1):
                entry = fused.setdefault(h["id"], [h, 0.0])
                entry[1] += 1.0 / (self.rrf_k + rank)
        top = sorted(fused.values(), key=lambda e: -e[1])[:k]
        return [dict(h, score=score) for h, score in top]


if __name__ == "__main__":
    query = " ".join(sys.argv[1:]) or "How do I declare query parameters?"
    chunks = load_chunks()
    bm25, dense = BM25Retriever(chunks), DenseRetriever(chunks)
    for retriever in (bm25, dense, HybridRetriever(bm25, dense)):
        print(f"\n=== {retriever.name} ===")
        for r in retriever.search(query, k=3):
            print(f"[{r['score']:.3f}] {r['source']} :: {r['heading']}")


In [ ]:
%%writefile src/evaluate.py
"""Evaluation helpers.

Retrieval eval (run from repo root):
    python src/evaluate.py
Compares BM25, the dense (transformer) retriever and a hybrid of both on data/questions.json using
Hit@k and MRR at page level: a retrieved chunk counts as correct if its source
file is one of the question's gold_sources.
"""
import csv
import json
import re
import sys
from collections import Counter
from pathlib import Path

sys.path.insert(0, str(Path(__file__).parent))
from retrieve import load_chunks, BM25Retriever, DenseRetriever, HybridRetriever  # noqa: E402

QUESTIONS = Path("data/questions.json")
RESULTS = Path("results")
KS = (1, 3, 5)
MAX_K = 10


def load_questions(path=QUESTIONS):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def first_rank(hits, gold):
    """1-based rank of the first retrieved chunk whose source is in gold, else None."""
    for i, h in enumerate(hits, 1):
        if h["source"] in gold:
            return i
    return None


def retrieval_eval(retriever, questions, ks=KS, max_k=MAX_K):
    rows, ranks = [], []
    for q in questions:
        if not q["answerable"]:
            continue
        hits = retriever.search(q["question"], k=max_k)
        r = first_rank(hits, set(q["gold_sources"]))
        ranks.append(r)
        rows.append({"id": q["id"], "question": q["question"], "gold": ";".join(q["gold_sources"]),
                     "rank": r if r else "", "top1_source": hits[0]["source"]})
    n = len(ranks)
    metrics = {f"hit@{k}": sum(1 for r in ranks if r and r <= k) / n for k in ks}
    metrics["mrr"] = sum(1 / r for r in ranks if r) / n
    metrics["n_questions"] = n
    return metrics, rows


# ---------- answer-quality helpers (used in the notebook) ----------
def _toks(s):
    return re.findall(r"\w+", s.lower())


def token_f1(pred, ref):
    p, r = Counter(_toks(pred)), Counter(_toks(ref))
    overlap = sum((p & r).values())
    if overlap == 0:
        return 0.0
    prec, rec = overlap / sum(p.values()), overlap / sum(r.values())
    return 2 * prec * rec / (prec + rec)


REFUSAL_PATTERNS = ("don't know", "do not know", "not in the context", "not mentioned",
                    "no information", "cannot answer", "can't answer", "not provided", "not found in")


def is_refusal(text):
    t = text.lower()
    return any(p in t for p in REFUSAL_PATTERNS)


def save_csv(path, rows):
    RESULTS.mkdir(exist_ok=True)
    if not rows:
        return
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        w.writeheader()
        w.writerows(rows)


def main():
    chunks = load_chunks()
    questions = load_questions()
    RESULTS.mkdir(exist_ok=True)
    summary = {}
    bm25, dense = BM25Retriever(chunks), DenseRetriever(chunks)
    for retriever in (bm25, dense, HybridRetriever(bm25, dense)):
        metrics, rows = retrieval_eval(retriever, questions)
        summary[retriever.name] = metrics
        save_csv(RESULTS / f"retrieval_{retriever.name}.csv", rows)
    (RESULTS / "retrieval_metrics.json").write_text(json.dumps(summary, indent=2))
    print(f"{'retriever':10s}" + "".join(f"{'hit@'+str(k):>8s}" for k in KS) + f"{'MRR':>8s}")
    for name, m in summary.items():
        print(f"{name:10s}" + "".join(f"{m[f'hit@{k}']:8.3f}" for k in KS) + f"{m['mrr']:8.3f}")
    print(f"(n = {next(iter(summary.values()))['n_questions']} answerable questions)")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile data/questions.json
[
  {
    "id": 1,
    "question": "How do I make a query parameter optional?",
    "reference_answer": "Give the parameter a default value of None, e.g. q: str | None = None.",
    "gold_sources": [
      "tutorial/query-params.md"
    ],
    "answerable": true
  },
  {
    "id": 2,
    "question": "If a route has skip=0 and limit=10 as defaults and I request /items/?skip=20, what values does my function get?",
    "reference_answer": "skip=20 (from the URL) and limit=10 (the default).",
    "gold_sources": [
      "tutorial/query-params.md"
    ],
    "answerable": true
  },
  {
    "id": 3,
    "question": "What exactly is the query part of a URL?",
    "reference_answer": "The set of key-value pairs after the ? in a URL, separated by & characters.",
    "gold_sources": [
      "tutorial/query-params.md"
    ],
    "answerable": true
  },
  {
    "id": 4,
    "question": "What response do I get if a path parameter declared as int receives the value foo?",
    "reference_answer": "An HTTP error with a JSON detail saying the input should be a valid integer (type int_parsing), pointing to path > item_id.",
    "gold_sources": [
      "tutorial/path-params.md"
    ],
    "answerable": true
  },
  {
    "id": 5,
    "question": "How do I declare the type of a path parameter?",
    "reference_answer": "Use standard Python type annotations on the function parameter, e.g. item_id: int.",
    "gold_sources": [
      "tutorial/path-params.md"
    ],
    "answerable": true
  },
  {
    "id": 6,
    "question": "How do I declare the JSON body that a client sends to my endpoint?",
    "reference_answer": "Create a Pydantic model that inherits from BaseModel and declare it as the type of a path operation function parameter.",
    "gold_sources": [
      "tutorial/body.md"
    ],
    "answerable": true
  },
  {
    "id": 7,
    "question": "Can a GET request carry a request body in FastAPI?",
    "reference_answer": "It is supported only for extreme cases but discouraged: behavior is undefined in the specs, Swagger UI will not document the body and proxies may not support it.",
    "gold_sources": [
      "tutorial/body.md"
    ],
    "answerable": true
  },
  {
    "id": 8,
    "question": "Which HTTP methods should I use to send data to my API?",
    "reference_answer": "POST (most common), PUT, DELETE or PATCH.",
    "gold_sources": [
      "tutorial/body.md"
    ],
    "answerable": true
  },
  {
    "id": 9,
    "question": "A client asks for an item that does not exist. How do I send back a not found response?",
    "reference_answer": "Raise an HTTPException with status_code=404 and a detail message.",
    "gold_sources": [
      "tutorial/handling-errors.md"
    ],
    "answerable": true
  },
  {
    "id": 10,
    "question": "Do I return or raise an HTTPException?",
    "reference_answer": "You raise it, because it is a normal Python exception; raising stops the rest of the path operation code.",
    "gold_sources": [
      "tutorial/handling-errors.md"
    ],
    "answerable": true
  },
  {
    "id": 11,
    "question": "Can the detail of an HTTPException be something other than a string?",
    "reference_answer": "Yes, any value convertible to JSON such as a dict or a list.",
    "gold_sources": [
      "tutorial/handling-errors.md"
    ],
    "answerable": true
  },
  {
    "id": 12,
    "question": "How can I run a slow job, like sending an email, after the response has already been returned?",
    "reference_answer": "Declare a BackgroundTasks parameter in the path operation function and call add_task() with the task function and its arguments.",
    "gold_sources": [
      "tutorial/background-tasks.md"
    ],
    "answerable": true
  },
  {
    "id": 13,
    "question": "What arguments does add_task receive?",
    "reference_answer": "The task function, then any positional arguments, then any keyword arguments to pass to that function.",
    "gold_sources": [
      "tutorial/background-tasks.md"
    ],
    "answerable": true
  },
  {
    "id": 14,
    "question": "When should I use Celery instead of BackgroundTasks?",
    "reference_answer": "For heavy background computation that does not need to share memory or variables with the app, or that must run in multiple processes or servers.",
    "gold_sources": [
      "tutorial/background-tasks.md"
    ],
    "answerable": true
  },
  {
    "id": 15,
    "question": "How do I let a frontend on a different domain call my API?",
    "reference_answer": "Add CORSMiddleware to the app with a list of allowed origins (allow_origins).",
    "gold_sources": [
      "tutorial/cors.md"
    ],
    "answerable": true
  },
  {
    "id": 16,
    "question": "Why does allow_origins=['*'] not work when my frontend sends cookies or Authorization headers?",
    "reference_answer": "A wildcard excludes anything involving credentials; origins, methods and headers must be specified explicitly when allow_credentials is True.",
    "gold_sources": [
      "tutorial/cors.md"
    ],
    "answerable": true
  },
  {
    "id": 17,
    "question": "What HTTP methods does CORSMiddleware allow by default?",
    "reference_answer": "Only GET.",
    "gold_sources": [
      "tutorial/cors.md"
    ],
    "answerable": true
  },
  {
    "id": 18,
    "question": "How long do browsers cache CORS responses by default?",
    "reference_answer": "600 seconds (max_age default).",
    "gold_sources": [
      "tutorial/cors.md"
    ],
    "answerable": true
  },
  {
    "id": 19,
    "question": "What counts as an origin in CORS?",
    "reference_answer": "The combination of protocol, domain and port; differences in any of them make a different origin.",
    "gold_sources": [
      "tutorial/cors.md"
    ],
    "answerable": true
  },
  {
    "id": 20,
    "question": "How do I accept a file upload from a client?",
    "reference_answer": "Declare a parameter with File or UploadFile (after installing python-multipart), since files are sent as form data.",
    "gold_sources": [
      "tutorial/request-files.md"
    ],
    "answerable": true
  },
  {
    "id": 21,
    "question": "Why would I use UploadFile instead of bytes for uploads?",
    "reference_answer": "UploadFile uses a spooled file (memory up to a limit then disk), so it handles large files, exposes metadata and has an async file-like interface.",
    "gold_sources": [
      "tutorial/request-files.md"
    ],
    "answerable": true
  },
  {
    "id": 22,
    "question": "What attributes does UploadFile expose?",
    "reference_answer": "filename, content_type and file (a SpooledTemporaryFile).",
    "gold_sources": [
      "tutorial/request-files.md"
    ],
    "answerable": true
  },
  {
    "id": 23,
    "question": "How do I change the HTTP status code of a response?",
    "reference_answer": "Pass the status_code parameter to the path operation decorator (e.g. @app.post(..., status_code=201)).",
    "gold_sources": [
      "tutorial/response-status-code.md"
    ],
    "answerable": true
  },
  {
    "id": 24,
    "question": "Which status code should I return after creating a new record?",
    "reference_answer": "201, Created.",
    "gold_sources": [
      "tutorial/response-status-code.md"
    ],
    "answerable": true
  },
  {
    "id": 25,
    "question": "How do I write automated tests for my FastAPI app?",
    "reference_answer": "Use TestClient (needs httpx) with pytest: create a client from the app, call it like httpx and assert on the response.",
    "gold_sources": [
      "tutorial/testing.md"
    ],
    "answerable": true
  },
  {
    "id": 26,
    "question": "Do my test functions need to be async?",
    "reference_answer": "No, test functions are normal def functions and calls to the client are normal calls without await.",
    "gold_sources": [
      "tutorial/testing.md"
    ],
    "answerable": true
  },
  {
    "id": 27,
    "question": "How do I serve static files such as images or CSS?",
    "reference_answer": "Import StaticFiles and mount a StaticFiles(directory=...) instance on a path such as /static.",
    "gold_sources": [
      "tutorial/static-files.md"
    ],
    "answerable": true
  },
  {
    "id": 28,
    "question": "What does it mean to mount an application?",
    "reference_answer": "Adding a complete independent application at a specific path that handles all sub-paths; its docs and OpenAPI are not part of the main app.",
    "gold_sources": [
      "tutorial/static-files.md"
    ],
    "answerable": true
  },
  {
    "id": 29,
    "question": "How do I write a function that runs for every request and response?",
    "reference_answer": "Use the @app.middleware('http') decorator on a function that receives request and call_next and returns the response.",
    "gold_sources": [
      "tutorial/middleware.md"
    ],
    "answerable": true
  },
  {
    "id": 30,
    "question": "If I add several middlewares, which one runs first?",
    "reference_answer": "The last middleware added is outermost and runs first on the request path and last on the response path.",
    "gold_sources": [
      "tutorial/middleware.md"
    ],
    "answerable": true
  },
  {
    "id": 31,
    "question": "How do I limit a query parameter to at most 50 characters?",
    "reference_answer": "Use Annotated[str | None, Query(max_length=50)].",
    "gold_sources": [
      "tutorial/query-params-str-validations.md"
    ],
    "answerable": true
  },
  {
    "id": 32,
    "question": "Can I validate a query parameter against a regular expression?",
    "reference_answer": "Yes, with the pattern parameter of Query.",
    "gold_sources": [
      "tutorial/query-params-str-validations.md"
    ],
    "answerable": true
  },
  {
    "id": 33,
    "question": "How do I require a path parameter to be an integer greater than or equal to 1?",
    "reference_answer": "Use Path(ge=1) in Annotated.",
    "gold_sources": [
      "tutorial/path-params-numeric-validations.md"
    ],
    "answerable": true
  },
  {
    "id": 34,
    "question": "How can I stop a user's password from appearing in the API response?",
    "reference_answer": "Declare an output model without the password field and use it as response_model (or return type) so FastAPI filters the data.",
    "gold_sources": [
      "tutorial/response-model.md"
    ],
    "answerable": true
  },
  {
    "id": 35,
    "question": "What happens if I declare both a return type annotation and response_model?",
    "reference_answer": "The response_model takes priority and is the one FastAPI uses.",
    "gold_sources": [
      "tutorial/response-model.md"
    ],
    "answerable": true
  },
  {
    "id": 36,
    "question": "How do I share logic like common query parameters between several endpoints?",
    "reference_answer": "Write a dependency function and declare it with Depends(), passing the function without calling it.",
    "gold_sources": [
      "tutorial/dependencies/index.md"
    ],
    "answerable": true
  },
  {
    "id": 37,
    "question": "How does FastAPI read headers like user-agent from a parameter named user_agent?",
    "reference_answer": "Header automatically converts underscores to hyphens; set convert_underscores=False to disable it.",
    "gold_sources": [
      "tutorial/header-params.md"
    ],
    "answerable": true
  },
  {
    "id": 38,
    "question": "Should I define my endpoints with async def or def?",
    "reference_answer": "Use async def if your libraries support await; otherwise use normal def. You can mix both and FastAPI handles them correctly.",
    "gold_sources": [
      "async.md"
    ],
    "answerable": true
  },
  {
    "id": 39,
    "question": "What do I need to install to receive form fields?",
    "reference_answer": "python-multipart, and then declare parameters with Form.",
    "gold_sources": [
      "tutorial/request-forms.md"
    ],
    "answerable": true
  },
  {
    "id": 40,
    "question": "What is the difference between PUT and PATCH for updating data?",
    "reference_answer": "PUT replaces the existing data; PATCH updates partially, typically using exclude_unset in model_dump to apply only the sent fields.",
    "gold_sources": [
      "tutorial/body-updates.md"
    ],
    "answerable": true
  },
  {
    "id": 41,
    "question": "What library does the docs example use for SQL databases?",
    "reference_answer": "SQLModel, built on SQLAlchemy and Pydantic; table models use table=True.",
    "gold_sources": [
      "tutorial/sql-databases.md"
    ],
    "answerable": true
  },
  {
    "id": 42,
    "question": "What does tokenUrl mean in OAuth2PasswordBearer?",
    "reference_answer": "The URL the client uses to send the username and password to get a token; requests then send Authorization: Bearer <token>.",
    "gold_sources": [
      "tutorial/security/first-steps.md"
    ],
    "answerable": true
  },
  {
    "id": 43,
    "question": "Where is the OpenAPI schema served by default and how can I change it?",
    "reference_answer": "At /openapi.json; change it with the openapi_url parameter.",
    "gold_sources": [
      "tutorial/metadata.md"
    ],
    "answerable": true
  },
  {
    "id": 44,
    "question": "How do I start the server from inside my Python file for debugging?",
    "reference_answer": "Import uvicorn and call uvicorn.run(app, host=..., port=...) inside an if __name__ == '__main__' block.",
    "gold_sources": [
      "tutorial/debugging.md"
    ],
    "answerable": true
  },
  {
    "id": 45,
    "question": "Who won the 2022 football World Cup?",
    "reference_answer": "NOT_IN_DOCS",
    "gold_sources": [],
    "answerable": false
  },
  {
    "id": 46,
    "question": "What is a good recipe for sourdough bread?",
    "reference_answer": "NOT_IN_DOCS",
    "gold_sources": [],
    "answerable": false
  },
  {
    "id": 47,
    "question": "How do I train a convolutional neural network on image data with PyTorch?",
    "reference_answer": "NOT_IN_DOCS",
    "gold_sources": [],
    "answerable": false
  }
]

## 2. Download the docs and (re)build the chunks

In [ ]:
import os
if not os.path.exists('data/fastapi'):
    !git clone --depth 1 https://github.com/fastapi/fastapi data/fastapi
!python src/ingest.py

## 3. Retrieval evaluation: BM25 vs. transformer (dense) vs. hybrid

In [ ]:
!python src/evaluate.py

In [ ]:
import json
import matplotlib.pyplot as plt

m = json.load(open('results/retrieval_metrics.json'))
names = [n for n in ('bm25', 'dense', 'hybrid') if n in m]
metrics = ['hit@1', 'hit@3', 'hit@5', 'mrr']
x = list(range(len(metrics))); w = 0.8 / len(names)
plt.figure(figsize=(8, 4))
for j, n in enumerate(names):
    plt.bar([i + (j - (len(names) - 1) / 2) * w for i in x], [m[n][k] for k in metrics], w, label=n)
plt.xticks(x, metrics); plt.ylim(0, 1); plt.ylabel('score'); plt.legend()
plt.title('Retrieval quality on the FastAPI question set')
plt.tight_layout(); plt.savefig('results/retrieval_comparison.png', dpi=150); plt.show()

## 4. Ablations

**4a. Which retrieval improvements help?** Baseline = every docs page, chunks without the page title. Improvements: drop non-documentation pages (the release notes were ~40% of all chunks), and add the page title to every chunk.

In [ ]:
import sys
import pandas as pd
sys.path.insert(0, 'src')
import ingest, retrieve
from evaluate import load_questions, retrieval_eval

questions = load_questions()
variants = {
    'baseline (all pages, no title)': dict(exclude=False, with_title=False),
    '+ exclude non-doc pages':        dict(exclude=True,  with_title=False),
    '+ page title in chunks':         dict(exclude=False, with_title=True),
    '+ both (final system)':          dict(exclude=True,  with_title=True),
}
rows = []
for label, kw in variants.items():
    ch = ingest.build_chunks(1200, **kw)
    b, d = retrieve.BM25Retriever(ch), retrieve.DenseRetriever(ch)
    for r in (b, d, retrieve.HybridRetriever(b, d)):
        met, _ = retrieval_eval(r, questions)
        rows.append({'variant': label, 'retriever': r.name, 'n_chunks': len(ch),
                     **{k: round(v, 3) for k, v in met.items() if k != 'n_questions'}})
abl_improve = pd.DataFrame(rows)
abl_improve.to_csv('results/ablation_improvements.csv', index=False)
abl_improve

**4b. Chunk size** (final system, max characters per chunk)

In [ ]:
rows = []
for size in (600, 1200, 2000):
    ch = ingest.build_chunks(size)
    b, d = retrieve.BM25Retriever(ch), retrieve.DenseRetriever(ch)
    for r in (b, d, retrieve.HybridRetriever(b, d)):
        met, _ = retrieval_eval(r, questions)
        rows.append({'max_chunk_chars': size, 'retriever': r.name, 'n_chunks': len(ch),
                     **{k: round(v, 3) for k, v in met.items() if k != 'n_questions'}})
abl_chunk = pd.DataFrame(rows)
abl_chunk.to_csv('results/ablation_chunk_size.csv', index=False)
abl_chunk

## 5. Load the generator (Qwen2.5-3B-Instruct)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL = 'Qwen/Qwen2.5-3B-Instruct'
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16, device_map='auto')
print('Model loaded on', model.device)

## 6. RAG pipeline

In [ ]:
from retrieve import load_chunks, BM25Retriever, DenseRetriever, HybridRetriever

chunks = load_chunks()
_b, _d = BM25Retriever(chunks), DenseRetriever(chunks)
retrievers = {'bm25': _b, 'dense': _d, 'hybrid': HybridRetriever(_b, _d)}

SYSTEM = ("You answer questions about the FastAPI documentation. "
          "Use ONLY the provided context. Only include code that appears in the context; "
          "never write your own code examples. "
          "If the answer is not in the context, say you don't know. "
          "End your answer with a line like 'Sources: [1], [3]' listing the context numbers you used.")

def answer(question, retriever=None, k=4, max_new_tokens=300):
    retriever = retriever or retrievers['dense']
    hits = retriever.search(question, k=k)
    context = "\n\n".join(
        f"[{i+1}] ({h['source']} :: {h['heading']})\n{h['text']}" for i, h in enumerate(hits))
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
    ]
    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    text = tok.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return text.strip(), hits

print(answer('How do I raise a 404 error?')[0])

## 7. Answer evaluation over the whole question set (takes several minutes)

In [ ]:
from tqdm.auto import tqdm
from evaluate import token_f1, is_refusal, first_rank, save_csv

all_rows = []
for name, retr in retrievers.items():
    for q in tqdm(questions, desc=name):
        ans, hits = answer(q['question'], retriever=retr, k=4)
        gold = set(q['gold_sources'])
        all_rows.append({
            'retriever': name, 'id': q['id'], 'answerable': q['answerable'],
            'question': q['question'], 'reference': q['reference_answer'], 'answer': ans,
            'sources': ' | '.join(h['source'] for h in hits),
            'gold_in_top4': (first_rank(hits, gold) is not None) if q['answerable'] else '',
            'token_f1': round(token_f1(ans, q['reference_answer']), 3) if q['answerable'] else '',
            'refused': is_refusal(ans),
            'human_correct': '', 'human_faithful': '',    # fill these in by hand (see section 8)
        })
save_csv('results/answers.csv', all_rows)

import pandas as pd
df = pd.DataFrame(all_rows)
ans_df = df[df.answerable].copy()
for col in ('token_f1', 'gold_in_top4', 'refused'):
    ans_df[col] = ans_df[col].astype(float)
unans_df = df[~df.answerable].copy()
unans_df['refused'] = unans_df['refused'].astype(float)
summary = pd.DataFrame({
    'gold_page_in_top4': ans_df.groupby('retriever').gold_in_top4.mean(),
    'mean_token_F1': ans_df.groupby('retriever').token_f1.mean(),
    'false_refusal_rate': ans_df.groupby('retriever').refused.mean(),
    'correct_refusal_rate (unanswerable)': unans_df.groupby('retriever').refused.mean(),
}).round(3)
summary.to_csv('results/answer_summary.csv')
summary

## 8. Human evaluation (do this by hand, ~15 min)

Open `results/answers.csv` from your Drive in Google Sheets. For the **dense** and **hybrid** rows, split the 47 questions between teammates and fill in:
- `human_correct`: 1 if the answer is correct compared with the reference, else 0
- `human_faithful`: 1 if every claim in the answer is supported by the retrieved sources, else 0

Token F1 is a rough metric for free-text answers, so these human scores are the numbers to report for answer quality.

## 9. Demo UI (Gradio)

In [ ]:
import gradio as gr

def ui_answer(question, retriever_name, k):
    ans, hits = answer(question, retriever=retrievers[retriever_name], k=int(k))
    src = "\n\n".join(
        f"**[{i}] {h['source']}** - {h['heading']}\n\n> " + h['text'][:300].replace("\n", " ") + "..."
        for i, h in enumerate(hits, 1))
    return ans, src

demo = gr.Interface(
    fn=ui_answer,
    inputs=[gr.Textbox(label='Question about FastAPI', lines=2),
            gr.Radio(['hybrid', 'dense', 'bm25'], value='hybrid', label='Retriever'),
            gr.Slider(1, 6, value=4, step=1, label='Top-k chunks')],
    outputs=[gr.Textbox(label='Answer'), gr.Markdown(label='Retrieved sources')],
    title='FastAPI Documentation QA (Transformer RAG)',
    examples=[['How do I raise a 404 error?', 'hybrid', 4],
              ['How do I allow a frontend on another domain to call my API?', 'hybrid', 4]],
)
demo.launch(share=True)